In [2]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

/home/tichar/Documents/ee798/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18144.96it/s]


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.

In [12]:
def mean_pool(hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def encode(model, inputs):
    with torch.no_grad():
        output = model(**inputs).last_hidden_state

    pooled = mean_pool(output, inputs["attention_mask"])
    return F.normalize(pooled, p=2, dim=1)


def encode_normal(text):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        output = model(**inputs).last_hidden_state

    pooled = mean_pool(output, inputs["attention_mask"])
    return F.normalize(pooled, p=2, dim=1)

def encode_recursive(text, T=1):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        z = model.embeddings(
            input_ids=inputs["input_ids"],
            token_type_ids=inputs.get("token_type_ids")
        )

        attention_mask = inputs["attention_mask"][:, None, None, :].to(z.dtype)
        attention_mask = (1.0 - attention_mask) * torch.finfo(z.dtype).min

        for _ in range(T):
            for layer in model.encoder.layer:
                z = layer(z, attention_mask=attention_mask)

    pooled = mean_pool(z, inputs["attention_mask"])
    return F.normalize(pooled, p=2, dim=1)

class SkipLayer(nn.Module):
    def forward(self, hidden_states, *args, **kwargs):
        return hidden_states



In [4]:
text = "Cross-Encoder achieve higher performance than Bi-Encoders, however, they do not scale well for large datasets. Here, it can make sense to combine Cross- and Bi-Encoders, for example in Info"
inputs = tokenizer(text, return_tensors="pt")

reference = encode(model, inputs)

print("Layer importance by skipping one layer:\n")

for i in range(len(model.encoder.layer)):
    test_model = copy.deepcopy(model)
    test_model.encoder.layer[i] = SkipLayer()
    test_model.eval()
    modified = encode(test_model, inputs)
    similarity = F.cosine_similarity(reference, modified).item()
    change = 1.0 - similarity
    print(f"skip layer {i}: cosine={similarity:.4f}, change={change:.4f}")

Layer importance by skipping one layer:

skip layer 0: cosine=0.6895, change=0.3105
skip layer 1: cosine=0.9285, change=0.0715
skip layer 2: cosine=0.9277, change=0.0723
skip layer 3: cosine=0.9357, change=0.0643
skip layer 4: cosine=0.8189, change=0.1811
skip layer 5: cosine=0.3003, change=0.6997


In [15]:
examples = examples = torch.load("commonsenseqa_50.pt", weights_only=False)

print(examples[0])

{'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?', 'candidates': ['bank', 'library', 'department store', 'mall', 'new york'], 'answer_key': 'A'}


In [16]:
label_to_index = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
Ts = [1, 2, 3, 4]

correct_counts = {T: 0 for T in Ts}
margin_sums = {T: 0.0 for T in Ts}

for ex in examples:
    correct_index = label_to_index[ex["answer_key"]]
    candidate_vecs = [encode_normal(c) for c in ex["candidates"]]

    for T in Ts:
        q = encode_recursive(ex["question"], T=T)

        scores = [F.cosine_similarity(q, c).item() for c in candidate_vecs]
        pred = max(range(len(scores)), key=lambda i: scores[i])

        if pred == correct_index:
            correct_counts[T] += 1

        correct_score = scores[correct_index]
        best_wrong = max(score for i, score in enumerate(scores) if i != correct_index)
        margin_sums[T] += correct_score - best_wrong


n = len(examples)

print(f"\nResults over {n} questions:\n")

for T in Ts:
    accuracy = correct_counts[T] / n
    avg_margin = margin_sums[T] / n

    print(f"T={T}: accuracy={accuracy:.3f}, avg_margin={avg_margin:.4f}")


Results over 50 questions:

T=1: accuracy=0.500, avg_margin=-0.0024
T=2: accuracy=0.420, avg_margin=-0.0253
T=3: accuracy=0.200, avg_margin=-0.0516
T=4: accuracy=0.200, avg_margin=-0.0568


In [21]:
def encode_refine(text, T=0):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        z = model.embeddings(
            input_ids=inputs["input_ids"],
            token_type_ids=inputs.get("token_type_ids")
        )

        attention_mask = inputs["attention_mask"][:, None, None, :].to(z.dtype)
        attention_mask = (1.0 - attention_mask) * torch.finfo(z.dtype).min

        # normal MiniLM pass
        for layer in model.encoder.layer:
            z = layer(z, attention_mask=attention_mask)

        # recursive refinement using only L4, L5
        for _ in range(T):
            z = model.encoder.layer[4](z, attention_mask=attention_mask)
            z = model.encoder.layer[5](z, attention_mask=attention_mask)

    pooled = mean_pool(z, inputs["attention_mask"])
    return F.normalize(pooled, p=2, dim=1)

In [22]:
Ts = [0, 1, 2, 3, 4]

correct_counts = {T: 0 for T in Ts}
margin_sums = {T: 0.0 for T in Ts}

for ex in examples:
    correct_index = label_to_index[ex["answer_key"]]
    candidate_vecs = [encode_normal(c) for c in ex["candidates"]]

    for T in Ts:
        q = encode_refine(ex["question"], T=T)

        scores = [F.cosine_similarity(q, c).item() for c in candidate_vecs]
        pred = max(range(len(scores)), key=lambda i: scores[i])

        if pred == correct_index:
            correct_counts[T] += 1

        correct_score = scores[correct_index]
        best_wrong = max(score for i, score in enumerate(scores) if i != correct_index)
        margin_sums[T] += correct_score - best_wrong

n = len(examples)

for T in Ts:
    accuracy = correct_counts[T] / n
    avg_margin = margin_sums[T] / n
    print(f"T={T}: accuracy={accuracy:.3f}, avg_margin={avg_margin:.4f}")

T=0: accuracy=0.500, avg_margin=-0.0024
T=1: accuracy=0.400, avg_margin=-0.0101
T=2: accuracy=0.360, avg_margin=-0.0384
T=3: accuracy=0.240, avg_margin=-0.0496
T=4: accuracy=0.200, avg_margin=-0.0520


In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()
N_LAYERS = len(model.encoder.layer)

/home/tichar/Documents/ee798/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18722.14it/s]


In [6]:
def mean_pool(h, mask):
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

def run_layer(layer, h, ext_mask):
    out = layer(h, attention_mask=ext_mask)
    return out[0] if isinstance(out, tuple) else out   # handles both transformers versions

@torch.no_grad()
def encode_custom(texts, layer_order, batch_size=64, max_length=128):
    """Encode texts by applying model.encoder.layer[i] for i in layer_order, in order."""
    outs = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt").to(device)
        h = model.embeddings(input_ids=enc["input_ids"],
                             token_type_ids=enc.get("token_type_ids"))
        # additive 4D mask: 0 for real tokens, large negative for padding
        ext = enc["attention_mask"][:, None, None, :].to(h.dtype)
        ext = (1.0 - ext) * torch.finfo(h.dtype).min
        for idx in layer_order:
            h = run_layer(model.encoder.layer[idx], h, ext)
        outs.append(F.normalize(mean_pool(h, enc["attention_mask"]), p=2, dim=1).cpu())
    return torch.cat(outs)

# Layer schedules
FULL = list(range(N_LAYERS))
def skip(i):            return [j for j in FULL if j != i]
def loop(a, b, k):      return FULL[:a] + FULL[a:b] * k + FULL[b:]   # repeat block [a, b) k times

In [7]:
# --- Data (mirror your Table I setup so the baseline row matches) ---
csqa = load_dataset("tau/commonsense_qa", split="validation")
questions = csqa["question"]
choices   = [t for c in csqa["choices"] for t in c["text"]]       # N*5, row-major
gold_csqa = torch.tensor(["ABCDE".index(a) for a in csqa["answerKey"]])

ag = load_dataset("fancyzhx/ag_news", split="test").shuffle(seed=0).select(range(2000))
ag_texts, gold_ag = ag["text"], torch.tensor(ag["label"])
AG_LABELS = ["World", "Sports", "Business", "Science and Technology"]  # use the same label strings as your main runs

def eval_csqa(order):
    q = encode_custom(questions, order)
    c = encode_custom(choices, order).view(len(questions), 5, -1)
    pred = (q.unsqueeze(1) * c).sum(-1).argmax(1)
    return (pred == gold_csqa).float().mean().item()

def eval_ag(order):
    x = encode_custom(ag_texts, order)
    lab = encode_custom(AG_LABELS, order)          # label embeddings go through the SAME modified model
    pred = (x @ lab.T).argmax(1)
    return (pred == gold_ag).float().mean().item()

In [8]:
# --- Sanity check: full schedule must reproduce the standard forward pass ---
t = ["a quick sanity check sentence"]
enc = tokenizer(t, return_tensors="pt").to(device)
with torch.no_grad():
    ref = F.normalize(mean_pool(model(**enc).last_hidden_state, enc["attention_mask"]), dim=1).cpu()
assert torch.allclose(ref, encode_custom(t, FULL), atol=1e-4), "custom forward differs from model forward"

# --- Experiments ---
configs = {"baseline": FULL}
for i in range(N_LAYERS):
    configs[f"skip_{i}"] = skip(i)
for (a, b) in [(1, 4), (2, 4), (1, 5)]:               # candidate "thinking" blocks
    for k in [2, 3, 4]:
        configs[f"loop_{a}-{b-1}_x{k}"] = loop(a, b, k)

rows = []
for name, order in configs.items():
    rows.append({"config": name, "n_layer_passes": len(order),
                 "csqa_acc": eval_csqa(order) * 100, "ag_acc": eval_ag(order) * 100})
    print(rows[-1])

res = pd.DataFrame(rows)
res["csqa_se"] = np.sqrt(res.csqa_acc/100 * (1 - res.csqa_acc/100) / len(questions)) * 100
res["ag_se"]   = np.sqrt(res.ag_acc/100   * (1 - res.ag_acc/100)   / len(ag_texts)) * 100
res.to_json("layer_ablation_minilm.json", orient="records", indent=2)
print(res.round(1))

{'config': 'baseline', 'n_layer_passes': 6, 'csqa_acc': 40.294840931892395, 'ag_acc': 63.65000009536743}
{'config': 'skip_0', 'n_layer_passes': 5, 'csqa_acc': 35.29893457889557, 'ag_acc': 53.200000524520874}
{'config': 'skip_1', 'n_layer_passes': 5, 'csqa_acc': 39.066338539123535, 'ag_acc': 63.70000243186951}
{'config': 'skip_2', 'n_layer_passes': 5, 'csqa_acc': 38.73873949050903, 'ag_acc': 66.00000262260437}
{'config': 'skip_3', 'n_layer_passes': 5, 'csqa_acc': 40.294840931892395, 'ag_acc': 63.89999985694885}
{'config': 'skip_4', 'n_layer_passes': 5, 'csqa_acc': 39.066338539123535, 'ag_acc': 58.799999952316284}
{'config': 'skip_5', 'n_layer_passes': 5, 'csqa_acc': 35.95413565635681, 'ag_acc': 52.95000076293945}
{'config': 'loop_1-3_x2', 'n_layer_passes': 9, 'csqa_acc': 39.55774009227753, 'ag_acc': 54.04999852180481}
{'config': 'loop_1-3_x3', 'n_layer_passes': 12, 'csqa_acc': 34.07043516635895, 'ag_acc': 42.59999990463257}
{'config': 'loop_1-3_x4', 'n_layer_passes': 15, 'csqa_acc': 27.